# Redes Neuronales (Funcion de activación y pérdida)

El dataset `make_moons` en sklearn.datasets es una **función** para generar datos sintéticos en forma de media luna, útil para probar **algoritmos de clasificación no lineal**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
X, y = make_classification(n_samples=1000, n_features=4, n_informative=3,
                           n_redundant=0, n_classes=3, random_state=42) #	Genera 4 caracteristicas (x1,x2,x3,x4), y(etiqueta) con 3 clases.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_cat = tf.keras.utils.to_categorical(y_train, 3)
y_test_cat = tf.keras.utils.to_categorical(y_test, 3)

# Malla para graficar fronteras de decisión
grid_x, grid_y = np.meshgrid(
    np.linspace(X_train_scaled[:,0].min()-1, X_train_scaled[:,0].max()+1, 200),
    np.linspace(X_train_scaled[:,1].min()-1, X_train_scaled[:,1].max()+1, 200)
)
grid = np.c_[grid_x.ravel(), grid_y.ravel()] #.ravel() es un método de NumPy que Convierte un arreglo de cualquier dimensión a un arreglo 1D (unidimensional).

**Diferencias entre input_dim vs input_shape en Keras**

input_dim:
* Sólo para capas `Dense`
* Número de características (dimensión) de entrada
* Un número entero
* Forma abreviada, menos general

input_shape:
* Para cualquier tipo de capa
* La forma completa del input sin incluir el tamaño del batch
* Una tupla (incluso si es 1D)
* Forma más flexible y preferida

In [ ]:
def build_model(activation):   # se pasa la variable activation para ser usada en diferentes capas de la red
    model = models.Sequential([
        layers.Dense(16, activation=activation, input_shape=(4,)),
        layers.Dense(16, activation=activation),
        layers.Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
activations = ['relu', 'tanh', 'sigmoid']
results = {}
histories = {}
models_dict = {}

for act in activations:
    print(f"Training con activation={act}...")
    model = build_model(act)
    history = model.fit(
        X_train_scaled, y_train_cat,
        epochs=100,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )
    histories[act] = history
    models_dict[act] = model
    loss, acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
    results[act] = acc
    print(f"Test Accuracy: {acc:.4f}\n")


In [ ]:
# Grafica de Validación Accuracy por Activación ('relu', 'tanh', 'sigmoid')
plt.figure(figsize=(8, 6))
for act in activations:
    plt.plot(histories[act].history['val_accuracy'], label=act, linewidth=2)
plt.title('Validation Accuracy por Activación', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
# Comparación de Accuracy en Test
plt.figure(figsize=(6, 6))
plt.bar(results.keys(), results.values())
plt.title('Test Accuracy Comparison', fontsize=14)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

# Gráfica de Fronteras

✅ Permite ver cómo el modelo separa las clases en el espacio de mayor varianza.

✅ Permite comparar diferentes funciones de activación (relu, tanh, sigmoid) y cómo afectan la frontera de decisión.

In [ ]:
from sklearn.decomposition import PCA

# Aplicar PCA para reducir a 2 dimensiones
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Crear malla en el espacio PCA
grid_x_pca, grid_y_pca = np.meshgrid(
    np.linspace(X_train_pca[:, 0].min()-1, X_train_pca[:, 0].max()+1, 200),
    np.linspace(X_train_pca[:, 1].min()-1, X_train_pca[:, 1].max()+1, 200)
)
grid_pca = np.c_[grid_x_pca.ravel(), grid_y_pca.ravel()]
grid_original = pca.inverse_transform(grid_pca)

# Graficar fronteras en espacio PCA
fig, axes = plt.subplots(1, len(activations), figsize=(18, 5))
for ax, act in zip(axes, activations):
    model = models_dict[act]
    Z = model.predict(grid_original)
    Z = np.argmax(Z, axis=1).reshape(grid_x_pca.shape)

    ax.contourf(grid_x_pca, grid_y_pca, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='bwr', edgecolor='k', s=20)
    ax.set_title(f'Activation: {act}', fontsize=14)
    ax.set_xlabel('PCA Component 1', fontsize=12)
    ax.set_ylabel('PCA Component 2', fontsize=12)
    ax.grid(True)

plt.tight_layout()
plt.show()


**¿Qué representa la gráfica?**

Es una visualización en 2D de cómo el modelo clasifica las diferentes clases del conjunto de datos después de aplicar reducción de dimensionalidad con PCA. Muestra:



1.   Fronteras de decisión: líneas que separan las regiones donde el modelo predice diferentes clases.
2.   Puntos de prueba (X_test): los datos reales, proyectados al nuevo espacio de 2 componentes principales.
3.   Colores:

     * Cada color en el fondo representa una clase que el modelo predice.

     * Cada punto muestra la clase real (y_test), permitiendo comparar predicción vs realidad.

**¿Qué hizo PCA?**

PCA transformó los 4 features originales en 2 componentes principales:

* Componente 1 y Componente 2: son nuevas variables que capturan la mayor varianza posible de los datos.

* Esto permite visualizar en 2D algo que originalmente tenía 4 dimensiones.